In [1]:
!pip install datasets

In [2]:
import json
import random
import os
from datasets import load_dataset

SEED = 42
random.seed(SEED)

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
dataset = load_dataset("sciq")

# Combine all splits
all_raw = (
    list(dataset["train"])
    + list(dataset["validation"])
    + list(dataset["test"])
)
print(f"Total raw samples: {len(all_raw)}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Total raw samples: 13679


In [4]:
def convert_to_instruction(sample):
    question = sample.get("question", "").strip()
    answer   = sample.get("correct_answer", "").strip()
    support  = sample.get("support", "").strip()

    if len(question) < 10:
        return None
    if len(answer) < 2:
        return None

    if support and len(support) > 20:
        response = f"{answer}. {support}"
    else:
        response = answer

    if len(response) < 20 or len(response) > 800:
        return None

    return {
        "instruction": question,
        "response"   : response
    }

def clean_dataset(raw_samples):
    converted = []
    seen_questions = set()
    skipped = 0

    for sample in raw_samples:
        result = convert_to_instruction(sample)

        if result is None:
            skipped += 1
            continue

        q_key = result["instruction"].lower().strip()
        if q_key in seen_questions:
            skipped += 1
            continue

        seen_questions.add(q_key)
        converted.append(result)

    print(f"Converted : {len(converted)}")
    print(f"Skipped   : {skipped}")
    return converted

clean = clean_dataset(all_raw)

Converted : 10405
Skipped   : 3274


In [5]:
random.shuffle(clean)

test  = clean[:200]
val   = clean[200:1200]
train = clean[1200:]

for name, split in [("train", train), ("validation", val), ("test", test)]:
    path = os.path.join(OUTPUT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(split, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(split)} samples → {path}")

Saved 9205 samples → data/train.json
Saved 1000 samples → data/validation.json
Saved 200 samples → data/test.json


In [6]:
with open("data/train.json") as f:
    data = json.load(f)

print(f"Train : {len(data)} samples")
print(f"\nSample:")
print(json.dumps(data[0], indent=2))

Train : 9205 samples

Sample:
{
  "instruction": "What biological agents that infect living hosts contain dna, yet lack the other parts shared by all cells, including a plasma membrane, cytoplasm, and ribosomes?",
  "response": "viruses. Viruses contain DNA but not much else. They lack the other parts shared by all cells, including a plasma membrane, cytoplasm, and ribosomes. Therefore, viruses are not cells, but are they alive? All living things not only have cells; they are also capable of reproduction. Viruses cannot reproduce by themselves. Instead, they infect living hosts, and use the hosts\u2019 cells to make copies of their own DNA. Viruses also do not have their own metabolism or maintain homeostasis. For these reasons, most scientists do not consider viruses to be living things."
}
